# Load Engine

In [19]:
import sys
import os
from pathlib import Path

# 1. Dynamically find the project root by searching for the folder name
def find_project_root(target_name="NCF-Recommender-System"):
    path = Path.cwd()
    for _ in range(5):  # Search up to 5 levels up
        if path.name == target_name:
            return path
        if (path / target_name).exists():
            return path / target_name
        path = path.parent
    raise FileNotFoundError(f"Could not find {target_name} in parent tree.")

try:
    root_dir = find_project_root()
    print(f"Project Root detected at: {root_dir}")
    
    # 2. Add to sys.path
    if str(root_dir) not in sys.path:
        sys.path.insert(0, str(root_dir)) # insert at 0 to prioritize this path

    # 3. CRITICAL: We MUST change directory for the internal loaders
    os.chdir(str(root_dir))

    # 4. Define paths relative to the newly set CWD
    META_PATH    = "research/datasets/processed/meta.json"
    WEIGHTS_PATH = "research/evaluation/pytorch_ncf_weights.pt"

    from research.hybrid.hybrid_engine import HybridEngine

    engine = HybridEngine.load(
        weights_path=META_PATH.replace("meta.json", "../../evaluation/pytorch_ncf_weights.pt"),
        meta_path=META_PATH
    )
    print("SUCCESS: Hybrid Engine loaded.")

except Exception as e:
    print(f"Error: {e}")

Project Root detected at: /Users/rahulkpkurup/Downloads/NCF-Recommender-System
SUCCESS: Hybrid Engine loaded.


## Test 1: cold-start user (0 interactions)

In [20]:
cold_recs = engine.recommend(user_id=0, seen_items=[], k=5)
print("Cold-start recommendations:")
for r in cold_recs:
    print(f"  item {r['item_id']:4d} | score {r['score']:.3f} | "
          f"source: {r['source']} | alpha: {r['alpha']}")

Cold-start recommendations:
  item 2651 | score 1.000 | source: cold_start | alpha: 0.0
  item  253 | score 0.832 | source: cold_start | alpha: 0.0
  item 1106 | score 0.819 | source: cold_start | alpha: 0.0
  item 1120 | score 0.799 | source: cold_start | alpha: 0.0
  item  466 | score 0.737 | source: cold_start | alpha: 0.0


## Test 2: warm user (simulate 15 interactions)

In [21]:
seen = [1, 5, 23, 50, 88, 120, 200, 310, 450, 600, 700, 800, 900, 1000, 1100]
warm_recs = engine.recommend(user_id=0, seen_items=seen, k=5)
print("\nWarm user recommendations:")
for r in warm_recs:
    print(f"  item {r['item_id']:4d} | score {r['score']:.3f} | "
          f"ncf: {r['ncf_score']:.3f} | cold: {r['cold_score']:.3f} | "
          f"alpha: {r['alpha']}")


Warm user recommendations:
  item  574 | score 0.956 | ncf: 0.973 | cold: 0.851 | alpha: 0.75
  item 1107 | score 0.909 | ncf: 0.904 | cold: 0.857 | alpha: 0.75
  item  144 | score 0.897 | ncf: 0.915 | cold: 0.830 | alpha: 0.75
  item  513 | score 0.890 | ncf: 0.915 | cold: 0.820 | alpha: 0.75
  item 1173 | score 0.870 | ncf: 0.991 | cold: 0.716 | alpha: 0.75


## Test 3: fully warm user (20+ interactions)

In [22]:
full_seen = list(range(1, 25))
full_recs = engine.recommend(user_id=1, seen_items=full_seen, k=5)
print("\nFully warm recommendations (NCF dominant):")
for r in full_recs:
    print(f"  item {r['item_id']:4d} | score {r['score']:.3f} | "
          f"source: {r['source']}")


Fully warm recommendations (NCF dominant):
  item  106 | score 1.000 | source: ncf
  item 1848 | score 1.000 | source: ncf
  item  339 | score 0.995 | source: ncf
  item  576 | score 0.995 | source: ncf
  item  513 | score 0.990 | source: ncf


## Test 4: similar item (search fallback)

In [23]:
similar = engine.recommend_similar(item_id=1, k=5)
print("\nItems similar to item 1:")
for r in similar:
    print(f"  item {r['item_id']:4d} | similarity: {r['score']:.3f}")


Items similar to item 1:
  item 1064 | similarity: 1.000
  item 2141 | similarity: 1.000
  item 2142 | similarity: 1.000
  item 2354 | similarity: 1.000
  item 2355 | similarity: 1.000
